# Exercise - Data Preparation

The data set for this exercise includes information on house sales in King County, Washington (between May 2014 and May 2015). (Each row in the data set pertains to one house. There is a total of 21,613 houses in the data set). Use the data set to practice your DATA PREP skills. 
<br><br>
The data set is called "kc_house_data.csv".  


## Description of Variables

The description and type of each variable is provided in "KC house data - Data Dictionary.docx". Make sure to read this document to learn about the variables. Please check if variables are numeric, binary, or categorical.

# Setup

In [1]:
# Common imports

from IPython import display
from base64 import b64decode
base64_data = "/9j/4AAQSkBQAUAFAH//2Q=="
display.Image(b64decode(base64_data))

import numpy as np
import pandas as pd

np.random.seed(17)

# Get the data

In [2]:
# Import the data set:

kchouse = pd.read_csv("kc_house_data.csv")
kchouse.head()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,432000.0,5.0,2.75,2060.0,329903.0,1.5,0,3,5,7.0,2060,0,1989.0,0,98022.0,47.1776,-121.944,2240,220232.0
1,170000.0,2.0,1.00,810.0,8424.0,1.0,0,0,4,6.0,810,0,1959.0,0,98023.0,47.3286,-122.346,820,8424.0
2,235000.0,3.0,1.00,960.0,5030.0,1.0,0,0,3,7.0,960,0,1955.0,0,98118.0,47.5611,-122.280,1460,5400.0
3,350000.0,2.0,1.00,830.0,5100.0,1.0,0,0,4,7.0,830,0,1942.0,0,98126.0,47.5259,-122.379,1220,5100.0
4,397380.0,2.0,1.00,1030.0,5072.0,1.0,0,0,3,6.0,1030,0,1924.0,1958,98115.0,47.6962,-122.294,1220,6781.0


# Split the data into train and test

In [3]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(kchouse, test_size=0.3)

## Check the missing values

In [4]:
train.isna().sum()

price            0
bedrooms         1
bathrooms        0
sqft_living      0
sqft_lot         1
floors           1
waterfront       0
view             0
condition        0
grade            1
sqft_above       0
sqft_basement    0
yr_built         1
yr_renovated     0
zipcode          1
lat              0
long             0
sqft_living15    0
sqft_lot15       0
dtype: int64

In [5]:
test.isna().sum()

price            0
bedrooms         0
bathrooms        0
sqft_living      1
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
grade            0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
zipcode          1
lat              0
long             0
sqft_living15    0
sqft_lot15       1
dtype: int64

In [6]:
train.shape, test.shape

((15129, 19), (6484, 19))

# Data Prep

In [7]:
# Imports:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

## Separate the PRICE variable, which is the target variable (don't transform the target)

In [8]:
# Separate the target variable and input variables

train_targets = train[['price']]
test_targets = test[['price']]

train_inputs = train.drop(['price'], axis=1)
test_inputs = test.drop(['price'], axis=1)

##  Identify the numeric, binary, and categorical columns

In [9]:
# Numerical
numeric_columns = train_inputs.select_dtypes(include=[np.number]).columns.to_list()
numeric_columns

['bedrooms',
 'bathrooms',
 'sqft_living',
 'sqft_lot',
 'floors',
 'waterfront',
 'view',
 'condition',
 'grade',
 'sqft_above',
 'sqft_basement',
 'yr_built',
 'yr_renovated',
 'zipcode',
 'lat',
 'long',
 'sqft_living15',
 'sqft_lot15']

In [10]:
# Categorical -- The automatic work did not recognize zip as a categorical
categorical_columns = ['zipcode']
categorical_columns

['zipcode']

In [11]:
# Binary
binary_columns = ['waterfront']
binary_columns

['waterfront']

In [12]:
for col in binary_columns:
    numeric_columns.remove(col)
    
for col in categorical_columns:
    numeric_columns.remove(col)

numeric_columns

['bedrooms',
 'bathrooms',
 'sqft_living',
 'sqft_lot',
 'floors',
 'view',
 'condition',
 'grade',
 'sqft_above',
 'sqft_basement',
 'yr_built',
 'yr_renovated',
 'lat',
 'long',
 'sqft_living15',
 'sqft_lot15']

# Pipeline (recommended)

If you don't want to use pipelines, feel free to use your own data prep steps.

In [13]:
# Numeric transformer:

numeric_transformer = Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])


In [14]:
# Categorical transformer:

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=99999)),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])


In [15]:
# Binary transformer:

binary_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))])


In [16]:
# Column transformer:

preprocessor = ColumnTransformer([
        ('num', numeric_transformer, numeric_columns),
        ('cat', categorical_transformer, categorical_columns),
        ('binary', binary_transformer, binary_columns)],
        remainder='passthrough')


# Transform: fit_transform() for TRAIN

In [17]:
#Fit and transform the train data

train_x = preprocessor.fit_transform(train_inputs)

train_x

<15129x88 sparse matrix of type '<class 'numpy.float64'>'
	with 257298 stored elements in Compressed Sparse Row format>

In [18]:
train_x.toarray()

array([[-0.3963457 , -0.46904059, -0.78887905, ...,  0.        ,
         0.        ,  0.        ],
       [-0.3963457 , -0.79341959, -0.79972134, ...,  0.        ,
         0.        ,  0.        ],
       [-1.45989912, -0.46904059, -0.94067105, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [-0.3963457 ,  0.50409642, -0.26844936, ...,  0.        ,
         0.        ,  0.        ],
       [-1.45989912,  1.47723343, -0.25760708, ...,  0.        ,
         0.        ,  0.        ],
       [-0.3963457 , -0.46904059,  0.4905106 , ...,  0.        ,
         0.        ,  0.        ]])

# Tranform: transform() for TEST

In [19]:
# Transform the test data

test_x = preprocessor.transform(test_inputs)

test_x

<6484x88 sparse matrix of type '<class 'numpy.float64'>'
	with 110286 stored elements in Compressed Sparse Row format>

In [20]:
test_x.toarray()

array([[-0.3963457 ,  0.50409642, -0.24676479, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.66720771, -0.14466159, -0.30097622, ...,  0.        ,
         0.        ,  0.        ],
       [-0.3963457 ,  0.17971742, -0.86477505, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.66720771,  0.50409642,  1.24947057, ...,  0.        ,
         0.        ,  0.        ],
       [-0.3963457 , -0.46904059, -0.95151333, ...,  0.        ,
         0.        ,  0.        ],
       [-1.45989912, -0.79341959, -0.64792935, ...,  0.        ,
         0.        ,  0.        ]])

In [21]:
test_x.shape

(6484, 88)